In [17]:
import sys
import os
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from pynwb import NWBHDF5IO
import h5py

from sklearn.decomposition import PCA


sys.path.append(os.path.join(os.getcwd(), '../import'))

# noinspection PyUnresolvedReferences
from utils import read_utils

In [18]:
path = os.path.join(os.path.expanduser('~'), '/lisc/scratch/neurobiology/zimmer/zimmermann/DATA/fly-2024-brezovec')			# path to nbw file folder
filename = 'sub-fly-087_ses-20200228161226_behavior+image+ophys.nwb'
filename_proc = 'sub-fly-087_ses-20200228161226_behavior.nwb'

In [19]:
def print_structure(name, _):
	print(name)

In [20]:
raw_file = h5py.File(os.path.join(path, filename), 'r')
raw_file.visititems(print_structure)

acquisition
acquisition/TwoPhotonSeriesAnatomicalGreen
acquisition/TwoPhotonSeriesAnatomicalGreen/data
acquisition/TwoPhotonSeriesAnatomicalGreen/dimension
acquisition/TwoPhotonSeriesAnatomicalGreen/field_of_view
acquisition/TwoPhotonSeriesAnatomicalGreen/starting_time
acquisition/TwoPhotonSeriesAnatomicalRed
acquisition/TwoPhotonSeriesAnatomicalRed/data
acquisition/TwoPhotonSeriesAnatomicalRed/dimension
acquisition/TwoPhotonSeriesAnatomicalRed/field_of_view
acquisition/TwoPhotonSeriesAnatomicalRed/starting_time
acquisition/TwoPhotonSeriesFunctionalGreen
acquisition/TwoPhotonSeriesFunctionalGreen/data
acquisition/TwoPhotonSeriesFunctionalGreen/dimension
acquisition/TwoPhotonSeriesFunctionalGreen/field_of_view
acquisition/TwoPhotonSeriesFunctionalGreen/starting_time
acquisition/TwoPhotonSeriesFunctionalRed
acquisition/TwoPhotonSeriesFunctionalRed/data
acquisition/TwoPhotonSeriesFunctionalRed/dimension
acquisition/TwoPhotonSeriesFunctionalRed/field_of_view
acquisition/TwoPhotonSeriesFunc

In [21]:
def print_sizes(name, obj):
    if isinstance(obj, h5py.Dataset):
        # Number of bytes = number of elements * size per element
        size = obj.size * obj.dtype.itemsize
            
        sizes.append((size, name))

sizes = []
    
raw_file.visititems(print_sizes)

# sort by size descending
sizes.sort(reverse=True)

for size, name in sizes[:50]:  # show top 50
    print(f"{size/1024/1024:.2f} MB  --  {name}")


53853.55 MB  --  processing/ophys/TwoPhotonSeriesFunctionalGreenProcessed/data
24100.00 MB  --  acquisition/TwoPhotonSeriesAnatomicalRed/data
24100.00 MB  --  acquisition/TwoPhotonSeriesAnatomicalGreen/data
10363.50 MB  --  acquisition/TwoPhotonSeriesFunctionalRed/data
10363.50 MB  --  acquisition/TwoPhotonSeriesFunctionalGreen/data
2.06 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationLabFrame/data
2.06 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationDeltaLabFrame/data
2.06 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationDeltaCameraFrame/data
2.06 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationCameraFrame/data
1.37 MB  --  processing/behavior/FicTrac/SpatialSeriesPosition/data
1.37 MB  --  processing/behavior/FicTrac/SpatialSeriesIntegratedMotion/data
0.69 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationDeltaError/data
0.69 MB  --  processing/behavior/FicTrac/SpatialSeriesMovementSpeed/data
0.69 MB  --  processing/behavior/FicTrac/SpatialSe

In [22]:
in_path = os.path.join(path, 'sub-fly-087_ses-20200228161226_behavior+image+ophys.nwb')
out_path = os.path.join(path, 'sub-fly-087_ses-20200228161226_behavior.nwb')

# which things to drop (example)
processing_to_drop = {"ophys"}

with NWBHDF5IO(in_path, mode="r", load_namespaces=True) as io_in:
	nwbfile = io_in.read()

	# --- drop acquisition TimeSeries by name ---
	for name in list(nwbfile.acquisition.keys()):
		del nwbfile.acquisition[name]
		
	# --- drop processing modules by name ---
	for name in list(nwbfile.processing.keys()):
		if name in processing_to_drop:
			del nwbfile.processing[name]

	# now write to a new file
	with NWBHDF5IO(out_path, mode="w") as io_out:
		io_out.export(io_in, nwbfile)


In [23]:
raw_file_proc = h5py.File(os.path.join(path, filename_proc), 'r')
raw_file_proc.visititems(print_structure)

acquisition
analysis
file_create_date
general
general/devices
general/devices/BrukerFluorescenceMicroscope
general/devices/Flea FL3-U3-13E4M-C
general/experiment_description
general/experimenter
general/institution
general/keywords
general/lab
general/optophysiology
general/optophysiology/ImagingPlaneFunctionalGreenProcessed
general/optophysiology/ImagingPlaneFunctionalGreenProcessed/Green
general/optophysiology/ImagingPlaneFunctionalGreenProcessed/Green/description
general/optophysiology/ImagingPlaneFunctionalGreenProcessed/Green/emission_lambda
general/optophysiology/ImagingPlaneFunctionalGreenProcessed/description
general/optophysiology/ImagingPlaneFunctionalGreenProcessed/excitation_lambda
general/optophysiology/ImagingPlaneFunctionalGreenProcessed/grid_spacing
general/optophysiology/ImagingPlaneFunctionalGreenProcessed/indicator
general/optophysiology/ImagingPlaneFunctionalGreenProcessed/location
general/optophysiology/ImagingPlaneFunctionalGreenProcessed/origin_coords
general/opt

In [24]:
sizes = []

def print_sizes(name, obj):
    if isinstance(obj, h5py.Dataset):
        # Number of bytes = number of elements * size per element
        size = obj.size * obj.dtype.itemsize
            
        sizes.append((size, name))
    
raw_file_proc.visititems(print_sizes)

# sort by size descending
sizes.sort(reverse=True)

for size, name in sizes[:50]:  # show top 50
    print(f"{size/1024/1024:.2f} MB  --  {name}")


2.06 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationLabFrame/data
2.06 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationDeltaLabFrame/data
2.06 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationDeltaCameraFrame/data
2.06 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationCameraFrame/data
1.37 MB  --  processing/behavior/FicTrac/SpatialSeriesPosition/data
1.37 MB  --  processing/behavior/FicTrac/SpatialSeriesIntegratedMotion/data
0.69 MB  --  processing/behavior/FicTrac/SpatialSeriesRotationDeltaError/data
0.69 MB  --  processing/behavior/FicTrac/SpatialSeriesMovementSpeed/data
0.69 MB  --  processing/behavior/FicTrac/SpatialSeriesMovementDirection/data
0.69 MB  --  processing/behavior/FicTrac/SpatialSeriesAnimalHeading/data
0.00 MB  --  general/keywords
0.00 MB  --  general/experimenter
0.00 MB  --  general/optophysiology/ImagingPlanetdTomatoFunctional/grid_spacing
0.00 MB  --  general/optophysiology/ImagingPlanetdTomatoAnatomical/grid_spacing
0.00 MB  --